In [2]:
import copy
from collections import deque

# parse board
def parse_board(lines):
    board = []
    for line in lines:
        row = [int(ch) for ch in line.strip() if ch.isdigit()]
        board.append(row)
    return board


def board_from_text(text):
    return parse_board(text.strip().splitlines())


def board_to_str(board):
    lines = []
    for r, row in enumerate(board):
        if r in (3, 6):
            lines.append("------+-------+------")
        parts = []
        for c, val in enumerate(row):
            if c in (3, 6):
                parts.append("|")
            parts.append(str(val) if val != 0 else ".")
        lines.append(" ".join(parts))
    return "\n".join(lines)

#  CSP REPRESENTATION

def get_peers(row, col):
    peers = set()
    # Same row
    for c in range(9):
        if c != col:
            peers.add((row, c))
    # Same column
    for r in range(9):
        if r != row:
            peers.add((r, col))
    # Same 3×3 box
    box_r, box_c = (row // 3) * 3, (col // 3) * 3
    for r in range(box_r, box_r + 3):
        for c in range(box_c, box_c + 3):
            if (r, c) != (row, col):
                peers.add((r, c))

    return peers

PEERS = {(r, c): get_peers(r, c) for r in range(9) for c in range(9)}


def build_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            val = board[r][c]
            if val != 0:
                domains[(r, c)] = {val}
            else:
                domains[(r, c)] = set(range(1, 10))
    return domains


def get_arcs():
    arcs = []
    for cell in [(r, c) for r in range(9) for c in range(9)]:
        for peer in PEERS[cell]:
            arcs.append((cell, peer))
    return arcs

#  AC-3 Arc Consistency Algorithm

def revise(domains, xi, xj):
    revised = False
    to_remove = set()
    for v in domains[xi]:
        if domains[xj] == {v}:
            to_remove.add(v)
    if to_remove:
        domains[xi] -= to_remove
        revised = True
    return revised

def ac3(domains):
    queue = deque(get_arcs())          # start with all arcs

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in PEERS[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True

#  FORWARD CHECKING

def forward_check(domains, var, value):
    for peer in PEERS[var]:
        if value in domains[peer]:
            domains[peer].discard(value)
            if len(domains[peer]) == 0:
                return False
    return True

#   VALUE ORDERING HEURISTICS

def select_unassigned_variable(domains):
    unassigned = [var for var, dom in domains.items() if len(dom) > 1]
    if not unassigned:
        return None

    # MRV: smallest domain
    min_size = min(len(domains[v]) for v in unassigned)
    candidates = [v for v in unassigned if len(domains[v]) == min_size]

    if len(candidates) == 1:
        return candidates[0]

    # Degree heuristic: most unassigned peers
    def degree(v):
        return sum(1 for p in PEERS[v] if len(domains[p]) > 1)

    return max(candidates, key=degree)

def order_domain_values(var, domains):
    def count_conflicts(val):
        return sum(1 for peer in PEERS[var] if val in domains[peer])

    return sorted(domains[var], key=count_conflicts)

#  BACKTRACKING SEARCH

def backtrack(domains, stats):
    stats['calls'] += 1

    if all(len(dom) == 1 for dom in domains.values()):
        return domains
        # select variable
    var = select_unassigned_variable(domains)
    if var is None:
        stats['failures'] += 1
        return None

    for value in order_domain_values(var, domains):
        new_domains = copy.deepcopy(domains)
        new_domains[var] = {value}

        # Forward checking
        if forward_check(new_domains, var, value):
            # AC-3 propagation
            if ac3(new_domains):
                result = backtrack(new_domains, stats)
                if result is not None:
                    return result
    stats['failures'] += 1
    return None


def solve(board):

    domains = build_domains(board)
    stats = {'calls': 0, 'failures': 0}
    if not ac3(domains):
        return None, stats

    result = backtrack(domains, stats)

    if result is None:
        return None, stats

    solution = [[next(iter(result[(r, c)])) for c in range(9)] for r in range(9)]
    return solution, stats


BOARDS = {
    "Easy": """
004030050
609400000
005100489
000060930
300807002
026040000
453009600
000004705
090050200
""",

    "Medium": """
005300000
800000020
070010500
400005300
010070006
003200080
060500009
004000030
000009700
""",

    "Hard": """
800000000
003600000
070090200
060005030
004800016
010300000
001006000
340080000
000007500
""",

    "Very Hard": """
000000000
000003085
001020000
000507000
004000100
090000000
500000073
002010000
000040009
"""
}


def main():
    all_results = {}

    for difficulty, text in BOARDS.items():
        board = board_from_text(text)

        print(f"  {difficulty} Board")
        print("Input:")
        print(board_to_str(board))

        solution, stats = solve(board)

        if solution:
            print("\nSolution:")
            print(board_to_str(solution))
        else:
            print("\nNo solution found!")

        print(f"\nStatistics:")
        print(f"  BACKTRACK calls   : {stats['calls']}")
        print(f"  BACKTRACK failures: {stats['failures']}")

        all_results[difficulty] = {
            'board': board,
            'solution': solution,
            'stats': stats
        }

    return all_results


if __name__ == "__main__":
    main()

  Easy Board
Input:
. . 4 | . 3 . | . 5 .
6 . 9 | 4 . . | . . .
. . 5 | 1 . . | 4 8 9
------+-------+------
. . . | . 6 . | 9 3 .
3 . . | 8 . 7 | . . 2
. 2 6 | . 4 . | . . .
------+-------+------
4 5 3 | . . 9 | 6 . .
. . . | . . 4 | 7 . 5
. 9 . | . 5 . | 2 . .

Solution:
7 8 4 | 9 3 2 | 1 5 6
6 1 9 | 4 8 5 | 3 2 7
2 3 5 | 1 7 6 | 4 8 9
------+-------+------
5 7 8 | 2 6 1 | 9 3 4
3 4 1 | 8 9 7 | 5 6 2
9 2 6 | 5 4 3 | 8 7 1
------+-------+------
4 5 3 | 7 2 9 | 6 1 8
8 6 2 | 3 1 4 | 7 9 5
1 9 7 | 6 5 8 | 2 4 3

Statistics:
  BACKTRACK calls   : 1
  BACKTRACK failures: 0
  Medium Board
Input:
. . 5 | 3 . . | . . .
8 . . | . . . | . 2 .
. 7 . | . 1 . | 5 . .
------+-------+------
4 . . | . . 5 | 3 . .
. 1 . | . 7 . | . . 6
. . 3 | 2 . . | . 8 .
------+-------+------
. 6 . | 5 . . | . . 9
. . 4 | . . . | . 3 .
. . . | . . 9 | 7 . .

Solution:
1 4 5 | 3 2 7 | 6 9 8
8 3 9 | 6 5 4 | 1 2 7
6 7 2 | 9 1 8 | 5 4 3
------+-------+------
4 9 6 | 1 8 5 | 3 7 2
2 1 8 | 4 7 3 | 9 5 6
7 5 3 | 2 9 6 | 4